# 05. Pipeline RAG Completo (Retrieval, Prompt Engineering & LLM)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:**
- Demonstração completa do fluxo: Pergunta -> Transformação -> Recuperação Semântica -> Contexto -> Geração com LLM -> Resposta fundamentada com Evidências.
- Respostas estritamente ancoradas nas evidências recuperadas.
- Tratamento de perguntas fora do escopo ou sem evidências suficientes (prevenção de alucinações).
- Exemplos de exploração da Voz do Cliente (reclamações, elogios, entrega, categorias).

In [ ]:
import os
import chromadb
from chromadb.utils import embedding_functions
from langchain.prompts import PromptTemplate
from dotenv import load_dotenv

load_dotenv()
DB_PATH = '../data/vector_db/'

# Conexão com a base vetorial persistida no ChromaDB
client = chromadb.PersistentClient(path=DB_PATH)
sentence_transformer_ef = embedding_functions.DefaultEmbeddingFunction()
collection = client.get_collection(name="olist_reviews", embedding_function=sentence_transformer_ef)
print(f"Base vetorial carregada com sucesso! Total de documentos: {collection.count():,}")

## 1. Configuração do Modelo de Linguagem (LLM)
Utilizamos o modelo **Google Gemini** (`gemini-1.5-flash`) via `langchain-google-genai` ou um mecanismo local/fallback caso a chave ainda não tenha sido configurada no arquivo `.env`.

In [ ]:
api_key = os.getenv("GOOGLE_API_KEY")
llm = None

if api_key and api_key != "SUA_CHAVE_AQUI":
    try:
        from langchain_google_genai import ChatGoogleGenerativeAI
        llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0.1, google_api_key=api_key)
        print("LLM Google Gemini inicializado com sucesso!")
    except Exception as e:
        print(f"Aviso: Falha ao inicializar Gemini: {e}")
else:
    print("Nota: GOOGLE_API_KEY não configurada no .env. O pipeline exibirá a montagem do contexto e o prompt.")

## 2. Construção do Prompt de Engenharia (Anti-Alucinação e Grounding)
O template de prompt impõe regras estritas de fidelidade às evidências e obriga o modelo a reconhecer quando não possui dados suficientes para responder.

In [ ]:
PROMPT_TEMPLATE = """
Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações de e-commerce.
Sua missão é responder à pergunta do gestor ESTRITAMENTE com base nas Evidências Recuperadas abaixo.

EVIDÊNCIAS RECUPERADAS DA BASE DE CONHECIMENTO:
{context}

DIRETRIZES OBRIGATÓRIAS:
1. FIDELIDADE AOS DADOS: Responda apenas com fatos sustentados pelos comentários acima. Nunca invente ou assuma informações externas.
2. RASTREABILIDADE E EVIDÊNCIAS: Cite trechos ou menções dos comentários que fundamentam cada insight apresentado.
3. TRATAMENTO DE INSUFICIÊNCIA: Se as evidências recuperadas forem insuficientes ou a pergunta estiver fora do escopo da base de clientes, responda EXATAMENTE:
   "Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta."

Pergunta do Gestor: {question}

Resposta Fundamentada do Analista:
"""

prompt = PromptTemplate(input_variables=["context", "question"], template=PROMPT_TEMPLATE)

## 3. Função Unificada do Pipeline RAG
A função `executar_pipeline_rag` orquestra a recuperação vetorial, a seleção de conteúdo, a formatação do contexto com metadados e a geração da resposta.

In [ ]:
def executar_pipeline_rag(pergunta: str, n_results: int = 5, where_filter: dict = None):
    """
    Executa o fluxo completo de Retrieval-Augmented Generation:
    1. Busca semântica no ChromaDB com métrica de cosseno.
    2. Extração dos documentos e IDs de evidência.
    3. Construção do contexto formatado.
    4. Geração da resposta com LLM e apresentação das evidências.
    """
    kwargs = {"query_texts": [pergunta], "n_results": n_results}
    if where_filter:
        kwargs["where"] = where_filter

    busca = collection.query(**kwargs)
    documentos = busca['documents'][0]
    ids = busca['ids'][0]
    metadados = busca['metadatas'][0]

    # Formatação do bloco de contexto
    blocos_contexto = []
    for i, (doc, doc_id, meta) in enumerate(zip(documentos, ids, metadados), 1):
        blocos_contexto.append(f"[Evidência {i} | ID: {doc_id} | Score: {meta.get('review_score')}]: {doc}")
    
    contexto_str = "\n".join(blocos_contexto)
    prompt_formatado = prompt.format(context=contexto_str, question=pergunta)

    print(f"\n{'='*70}")
    print(f"PERGUNTA: {pergunta}")
    print(f"{'='*70}")
    print("\n--- EVIDÊNCIAS RECUPERADAS (Top Evidências) ---")
    for b in blocos_contexto:
        print(b)

    print("\n--- RESPOSTA FUNDAMENTADA DO SISTEMA RAG ---")
    if llm:
        resposta = llm.invoke(prompt_formatado)
        print(resposta.content)
    else:
        print("[Simulação / Pré-visualização do Prompt]:")
        print(f"O contexto com {len(documentos)} evidências foi preparado para envio ao modelo de linguagem.")
    print(f"{'='*70}\n")

## 4. Testes e Exemplos de Exploração da Voz do Cliente (Edital do Desafio)
Vamos rodar os cenários solicitados no edital:

In [ ]:
# Cenário 1: Investigação de Reclamações de Entrega (com filtro de nota <= 2)
executar_pipeline_rag(
    pergunta="O que os clientes relatam sobre problemas relacionados à entrega?",
    n_results=4,
    where_filter={"review_score": {"$lte": 2}}
)

# Cenário 2: Busca por Experiências Positivas e Elogios (com filtro de nota >= 4)
executar_pipeline_rag(
    pergunta="Quais aspectos da experiência de compra são mais elogiados pelos clientes?",
    n_results=4,
    where_filter={"review_score": {"$gte": 4}}
)

## 5. Tratamento de Perguntas sem Evidências Suficientes / Fora do Escopo
Conforme o item 4 do edital, soluções baseadas em IA podem produzir respostas aparentemente convincentes mesmo sem evidências. O sistema deve reconhecer limitações e recusar a responder quando a pergunta estiver fora do domínio.

In [ ]:
# Cenário 3: Pergunta completamente fora de escopo dos dados de e-commerce
executar_pipeline_rag(
    pergunta="Qual é a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?",
    n_results=3
)